# Ridge & Lasso Regularization Paths

Wiki reference for [ridge & lasso paths](https://ml-viz-ruby.vercel.app/wiki/ridge-lasso-paths).

**The idea in one sentence.** Both ridge ($L_2$) and lasso ($L_1$) shrink coefficients toward
zero as $\lambda$ grows, but the *shape* of the penalty differs: ridge shrinks **smoothly and
never to exactly zero** (spreading weight across correlated features), while lasso drives
coefficients to **exact zero** — doing feature selection.

We implement ridge from scratch and trace both regularization paths, **validate the shrinkage
and lasso's sparsity**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, lasso_path
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  1.5,
})

np.random.seed(42)

## 1 — Ridge closed-form from scratch

$$\mathbf{w}^*_\text{Ridge} = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}$$

In [ ]:
def ridge_closed_form(X, y, lam):
    n, p = X.shape
    return np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)

# 3-point sanity check from the wiki worked example
X_small = np.array([[1, 1], [2, 1], [3, 1]], dtype=float)
y_small = np.array([1.0, 2.0, 2.0])
X_small_centered = X_small - X_small.mean(0)

w0 = ridge_closed_form(X_small_centered, y_small, lam=1e-6)  # OLS (tiny lam: the centered constant column is singular at lam=0)
w1 = ridge_closed_form(X_small_centered, y_small, lam=1.0)
w5 = ridge_closed_form(X_small_centered, y_small, lam=5.0)

print(f"λ→0 (OLS):  w = {w0}")
print(f"λ=1:        w = {w1}")
print(f"λ=5:        w = {w5}")
print("Ridge shrinks weights toward 0 as λ increases.")

### Validate: ridge shrinks the weights as $\lambda$ grows

The ridge penalty $\lambda\|w\|^2$ pulls coefficients toward zero, so a larger $\lambda$ gives a
smaller weight norm. We confirm $\lambda=1$ shrinks the weights relative to (near-)OLS.

In [ ]:
print(f'||w||: OLS(lam->0) = {np.linalg.norm(w0):.3f},  ridge(lam=1) = {np.linalg.norm(w1):.3f},  ridge(lam=5) = {np.linalg.norm(w5):.3f}')
assert np.linalg.norm(w1) < np.linalg.norm(w0), 'ridge (lam>0) shrinks the weights vs OLS'
assert np.linalg.norm(w5) < np.linalg.norm(w1), 'more regularization shrinks further'
print('\n✅ ridge shrinks coefficients smoothly toward zero as lambda grows')

## 2 — Synthetic 10-feature dataset

3 strong, 2 medium, 1 correlated echo, 3 pure noise features.

In [ ]:
n, p = 200, 10
X = np.random.randn(n, p)
X[:, 7] = X[:, 0] + 0.2 * np.random.randn(n)   # feature 7 echoes feature 0
true_w = np.array([3, 2, 1.5, 0, 0, 0.8, 0.5, 0, 0, 0])
y = X @ true_w + np.random.randn(n)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

print("True weights:", true_w)
print("Feature 7 is a noisy echo of feature 0 — Ridge will shrink both.")

## 3 — Coefficient paths: Ridge vs. Lasso

In [ ]:
lambdas = np.logspace(-3, 2, 120)

coefs_ridge = np.array([ridge_closed_form(X_s, y, lam) for lam in lambdas])
coefs_lasso = np.array([Lasso(alpha=lam, max_iter=10000).fit(X_s, y).coef_
                        for lam in lambdas])

colors = ['#6366f1','#20d9d2','#f97316','#ef4444','#a78bfa',
          '#34d399','#fbbf24','#fb7185','#60a5fa','#94a3b8']
feature_labels = [f'w{j} (true={true_w[j]:.1f})' for j in range(p)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for j in range(p):
    lbl = feature_labels[j] if true_w[j] != 0 else None
    ax1.semilogx(lambdas, coefs_ridge[:, j], color=colors[j], lw=1.5, label=lbl)
    ax2.semilogx(lambdas, coefs_lasso[:, j], color=colors[j], lw=1.5)

ax1.set_title('Ridge: smooth shrinkage, never exactly zero')
ax1.set_xlabel('λ (log scale)'); ax1.set_ylabel('Coefficient')
ax1.axhline(0, color='#555', lw=0.5); ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=8)

ax2.set_title('Lasso: coefficients reach exactly zero')
ax2.set_xlabel('λ (log scale)')
ax2.axhline(0, color='#555', lw=0.5); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Validate: lasso produces exact zeros; ridge does not

At the same $\lambda$, lasso zeros out several coefficients (feature selection), while ridge
shrinks every coefficient but keeps them all non-zero. We confirm the sparsity contrast at a
mid-range $\lambda$.

In [ ]:
mid = len(lambdas) // 2
n_zero_lasso = int((coefs_lasso[mid] == 0).sum())
n_zero_ridge = int((np.abs(coefs_ridge[mid]) < 1e-8).sum())
print(f'at lambda={lambdas[mid]:.3f}: lasso zeros = {n_zero_lasso}, ridge zeros = {n_zero_ridge}')
assert n_zero_lasso > n_zero_ridge, 'lasso (L1) zeros coefficients (sparse); ridge (L2) shrinks but keeps them'
print('\n✅ L1 selects features (exact zeros); L2 shrinks all of them')

## 4 — Feature elimination order in Lasso

In [ ]:
alphas, coefs_path, _ = lasso_path(X_s, y, n_alphas=200)

print("Feature elimination order (weakest first):")
deaths = []
for j in range(p):
    zeros = np.where(coefs_path[j] == 0)[0]
    dead_at = alphas[zeros[0]] if len(zeros) else None
    deaths.append((dead_at if dead_at else 0.0, j))
    status = f"zeroed at λ={dead_at:.4f}" if dead_at else "never zeroed"
    print(f"  w{j} (true={true_w[j]:.1f}): {status}")

# Bar chart: λ at which each feature dies
death_lams = [d[0] for d in sorted(deaths)]
feature_idx = [d[1] for d in sorted(deaths)]

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.bar([f'w{j}' for j in feature_idx], death_lams,
              color=[colors[j] for j in feature_idx], alpha=0.85)
ax.set_xlabel('Feature')
ax.set_ylabel('λ at elimination')
ax.set_title('Lasso feature elimination threshold (higher = more important)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5 — Cross-validated λ selection

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge_cv.fit(X_s, y)

lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_s, y)

print(f"Ridge CV best λ:   {ridge_cv.alpha_:.4f}")
print(f"Ridge CV coefficients: {ridge_cv.coef_.round(3)}")
print(f"\nLasso CV best λ:   {lasso_cv.alpha_:.4f}")
print(f"Lasso nonzero features: {np.sum(lasso_cv.coef_ != 0)} / {p}")
print(f"Lasso CV coefficients: {lasso_cv.coef_.round(3)}")

# Compare to true weights
fig, ax = plt.subplots(figsize=(9, 3))
x_pos = np.arange(p)
ax.bar(x_pos - 0.25, true_w, width=0.25, color='#94a3b8', alpha=0.8, label='True')
ax.bar(x_pos,        ridge_cv.coef_, width=0.25, color='#6366f1', alpha=0.8, label='Ridge CV')
ax.bar(x_pos + 0.25, lasso_cv.coef_, width=0.25, color='#f97316', alpha=0.8, label='Lasso CV')
ax.set_xticks(x_pos); ax.set_xticklabels([f'w{j}' for j in range(p)])
ax.set_title('True vs. Ridge CV vs. Lasso CV coefficients')
ax.axhline(0, color='#555', lw=0.5)
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **not standardizing** | penalties are scale-dependent — standardize features first |
| **lasso under correlation** | arbitrarily picks one of a correlated group (demo) — unstable |
| **ridge never selects** | it keeps all features (verified) — no sparsity |
| **choosing lambda** | tune by cross-validation, not by eye |
| **penalizing the intercept** | usually exclude the bias from the penalty |

Demo: ridge spreads weight across correlated features; lasso concentrates it.

In [ ]:
# The behaviour on CORRELATED features (feature 7 is a noisy echo of feature 0): ridge SPREADS
# weight across the correlated pair, keeping meaningful weight on both; lasso CONCENTRATES it,
# pushing most onto one and shrinking the other toward zero (eliminating it entirely at larger
# lambda). We compare the echo-feature weight under each.
from sklearn.linear_model import Lasso
w_ridge = ridge_closed_form(X_s, y, lam=1.0)
w_lasso = Lasso(alpha=0.1, max_iter=10000).fit(X_s, y).coef_
print(f'echo feature (w7): ridge {w_ridge[7]:.3f}  vs lasso {w_lasso[7]:.3f}')
assert abs(w_ridge[7]) > abs(w_lasso[7]), 'ridge spreads weight onto the correlated echo feature; lasso concentrates it away'
print('\nRidge shares weight among correlated features; lasso arbitrarily picks one -> unstable selection under correlation.')

## ✏️ Your turn

### Exercise 1 — Ridge in the eigenbasis

The wiki shows that in the eigenbasis of $X^\top X$ with eigenvalues $\sigma_j^2$,
Ridge multiplies each component by $\sigma_j^2/(\sigma_j^2+\lambda)$.

Verify this numerically for `X_s` at $\lambda = 1$: compute the SVD of $X_s$,
reconstruct the Ridge solution using the shrinkage formula, and compare to
`ridge_closed_form(X_s, y, lam=1.0)`.

In [ ]:
# TODO(you): verify Ridge via SVD shrinkage
lam = 1.0

# U, s, Vt = np.linalg.svd(X_s, full_matrices=False)
# sigma2 = s**2
# shrinkage = sigma2 / (sigma2 + lam)
# w_svd = ???
# w_direct = ridge_closed_form(X_s, y, lam)
# print("SVD formula:", w_svd.round(4))
# print("Direct solve:", w_direct.round(4))
# print("Match:", np.allclose(w_svd, w_direct, atol=1e-6))

### Exercise 2 — Elastic Net path

The Elastic Net combines Ridge and Lasso penalties:
$$\mathcal{L}(\mathbf{w}) = \|y - X\mathbf{w}\|^2 + \lambda_1\|\mathbf{w}\|_1 + \lambda_2\|\mathbf{w}\|^2$$

Use `sklearn.linear_model.ElasticNet` with `l1_ratio=0.5` to plot the coefficient
path on `X_s, y`. Compare sparsity to the pure Lasso path.

<details>
<summary>Solution outline</summary>

```python
from sklearn.linear_model import ElasticNet

coefs_en = np.array([ElasticNet(alpha=lam, l1_ratio=0.5, max_iter=10000).fit(X_s, y).coef_
                     for lam in lambdas])

fig, ax = plt.subplots(figsize=(8, 4))
for j in range(p):
    ax.semilogx(lambdas, coefs_en[:, j], color=colors[j], lw=1.5)
ax.axhline(0, color='#555', lw=0.5)
ax.set_title('Elastic Net path (l1_ratio=0.5)')
ax.set_xlabel('λ'); ax.set_ylabel('Coefficient')
ax.grid(True, alpha=0.3); plt.show()
```
</details>

## Key takeaways

- **Both shrink toward zero** as $\lambda$ grows (verified for ridge).
- **Lasso ($L_1$) gives exact zeros** — feature selection; ridge ($L_2$) shrinks but keeps all
  (verified).
- **Correlated features:** ridge spreads weight across them; lasso picks one (demo).
- **Choose by goal:** lasso for sparsity/selection, ridge for stability under correlation
  (elastic net blends both).